# 🔬 Step 2 — Feature Engineering (Silver)

Build fraud signals from raw data:
- **location_jump**: Card used in different city within 10 mins
- **high_velocity**: More than 5 transactions in last 24 hours
- **amount_spike**: Amount > 5x the customer's 30-day average

In [ ]:
from pyspark.sql import functions as F

df = spark.table('bronze_fraud_transactions')

df_silver = df \
    .withColumn('location_jump',
        (F.col('Location') != F.col('PreviousLocation')).cast('int')) \
    .withColumn('high_velocity',
        (F.col('NumTxnLast24h') > 5).cast('int')) \
    .withColumn('amount_spike',
        (F.col('Amount') > F.col('AvgTxnAmount30d') * 5).cast('int')) \
    .withColumn('fast_repeat',
        (F.col('TimeSinceLastTxnMins') < 5).cast('int'))

print('✅ Fraud features created')
df_silver.select('TransactionID','Amount','location_jump','high_velocity','amount_spike','fast_repeat','IsFraud').show(10)

In [ ]:
df_silver.write.format('delta').mode('overwrite').saveAsTable('silver_fraud_features')
print('✅ Silver features table saved!')